# ATST add-on cells for the V5 kernel (run top to bottom)

Run these **in the kernel where the V5 notebook already ran** (same session: `VALN`, `VALS`, `tracks_v5`, `TRAIN_ROWS`, ... must exist).
Nothing here re-runs V5 cells or restarts the kernel. Cells 1-6 write nothing but the test file and are safe.
The experiment cells (7-12) are gated by `GO_*` flags that default to `False`, so *Run All* never starts hours of work.
Prerequisite: `atst_common.py`, `atst_v5_integration.py`, `vaani_track_ensemble.py` are already in `BASE_DIR/atst_work` (unchanged).

In [ ]:
# ---- CELL 1: setup, guards, gates (the only cell with settings) ----------------------------------------------------
import os, sys, json, shutil, importlib, numpy as np, torch
assert all(n in globals() for n in ("VaaniSEDV5", "VALN", "VALS", "tracks_v5", "TRAIN_ROWS", "CFG_V5", "select_decoder", "V5_DIR")), \
    "V5 objects are not in this kernel. Run this inside the kernel where the V5 notebook ran."
HERE = f"{BASE_DIR}/atst_work"
CKPT = f"{BASE_DIR}/atst_as2M.ckpt"
AUDIOSSL = f"{BASE_DIR}/audiossl"          # optional clone of Audio-WestlakeU/audiossl (enables the reference-equivalence tests)
for f in ("atst_common.py", "atst_v5_integration.py", "vaani_track_ensemble.py"):
    assert os.path.isfile(f"{HERE}/{f}"), f"missing {HERE}/{f}"
assert os.path.isfile(CKPT), CKPT
if HERE not in sys.path: sys.path.insert(0, HERE)
_t = f"{HERE}/test_atst_common.py"
if os.path.exists(_t) and not os.path.exists(_t + ".bak"): shutil.copy(_t, _t + ".bak")

# experiment gates: turn ON one at a time, in this order
GO_PROBE_CACHE = False   # cell 7  : build probe cache (Gold + Val-S + 3000 synth)
GO_PROBE       = False   # cell 8  : BEATs vs ATST vs both frozen-feature probe (go/no-go)
GO_ENSEMBLE    = False   # cell 9  : Experiment A, CPU only
GO_FULL_CACHE  = False   # cell 10 : full ATST cache (only if the probe passed)
GO_TRAIN       = False   # cell 11 : B3 fusion training (~2.5 h)
GO_EVAL        = False   # cell 12 : B3 evaluation vs V5-C under the fixed V5-A decoder
GO_TEST_ZIP    = False   # cell 13 : B3 test tracks + zip (only if cell 12 said ACCEPT)
print("kernel SMOKE =", globals().get("SMOKE"), "| V5_DIR =", V5_DIR, "| device =", device)

In [ ]:
%%writefile C:/Users/231b258/atst_work/test_atst_common.py
"""test_atst_common.py (v2, corrected). Use from a notebook: import test_atst_common as T; T.test_...()"""
import os, sys, tempfile, types
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import atst_common as AC

torch.manual_seed(0)


def rand_encoder(seed=0):
    torch.manual_seed(seed)
    net = AC.FrameASTMin()
    for name, p in net.named_parameters():
        if p.ndim > 1:
            torch.nn.init.normal_(p, mean=0.0, std=0.02)
        elif name.endswith(".weight"):
            torch.nn.init.ones_(p)            # LayerNorm scales (zero would collapse every output to 0)
        else:
            torch.nn.init.zeros_(p)
    net.requires_grad_(False)
    net.eval()
    return net


def test_rand_encoder_nontrivial():
    net = rand_encoder()
    with torch.inference_mode():
        y = net(AC.ATSTMel()(torch.randn(1, 16000 * 4) * 0.1))
    assert torch.isfinite(y).all() and y.std().item() > 1e-3, float(y.std())
    print("ok rand_encoder: finite, std %.3f" % y.std().item())


def test_mel_per_example_floor():
    m = AC.ATSTMel("pretrain")
    a, b = torch.randn(1, 32000) * 0.3, torch.randn(1, 32000) * 0.003
    batch = m(torch.cat([a, b]))
    assert torch.allclose(batch[0], m(a)[0], atol=1e-5) and torch.allclose(batch[1], m(b)[0], atol=1e-5), \
        "top_db floor leaked across batch items"
    assert batch.shape == (2, 1, 64, 1 + 32000 // 160)
    print("ok mel: per-example floor, shape", tuple(batch.shape))


def test_zscore_shifts_levels():
    m = AC.ATSTMel("pretrain"); x = torch.randn(1, 64000) * 0.01
    z = (x - x.mean()) / (x.std() + 1e-5)
    d = float(m(z).mean() - m(x).mean())
    print(f"info: per-clip z-scoring moves the MinMax-normalised mel mean by {d:+.3f} (scale range is 2.0)")
    assert abs(d) > 0.1


def test_padding_mask_equivalence():
    net = rand_encoder()
    mel = AC.ATSTMel()(torch.randn(2, 16000 * 6) * 0.1)            # (2,1,64,601)
    mel = mel[..., : (mel.shape[-1] // 4) * 4]                      # 600 frames = 150 tokens (patch embed drops the rest)
    short = mel[:1, :, :, :4 * 100]
    with torch.inference_mode():
        alone = net(short, torch.tensor([100]))
        padded = torch.cat([torch.nn.functional.pad(short, (0, 4 * 50)), mel[1:]], dim=0)
        both = net(padded, torch.tensor([100, 150]))
    err = float((alone[0] - both[0, :100]).abs().max())
    assert err < 1e-4, err
    print("ok padded batch == unbatched (max err %.2e)" % err)


def test_chunk_plan_coverage():
    for n_tok in (1, 249, 250, 251, 375, 500, 501, 642, 1000):
        plan = AC.plan_chunks(n_tok)
        cov = np.zeros(n_tok, np.int32)
        for s, e, k0, k1 in plan:
            assert 0 <= s < e <= n_tok and s <= k0 < k1 <= e and e - s <= AC.MAX_TOKENS
            cov[k0:k1] += 1
        assert np.all(cov == 1), (n_tok, np.where(cov != 1)[0][:5])
        assert plan[0][2] == 0 and plan[-1][3] == n_tok
    print("ok chunk plan: no gaps, no duplicate kept tokens")


def test_extractor_shapes_and_chunking():
    enc = AC.ATSTEncoder(rand_encoder(), AC.ATSTMel(), {"embed_dim": 768})
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    for sec in (0.4, 1.0, 7.9, 8.0, 10.0, 10.3, 25.7):
        w = torch.randn(int(sec * AC.SR)) * 0.05
        f = ex(w)
        exp = AC.atst_T_expected(max(w.numel(), AC.SR))
        assert f.shape == (exp, 768) and np.isfinite(f).all(), (sec, f.shape, exp)
    w = torch.randn(int(9.0 * AC.SR)) * 0.05
    with torch.inference_mode():
        direct = enc.net(enc.mel(w[None]), torch.tensor([225]))[0].cpu().numpy()
    assert np.allclose(ex(w), direct, atol=1e-4)
    print("ok extractor: shapes for 0.4-25.7 s, single-chunk == direct")


_PREFIX = {"c2f_frame_encoder": "encoder.encoder.frame_encoder.", "lightning_teacher": "model.teacher.encoder.",
           "downstream_finetuned": "encoder.encoder.", "pretrainedsed_wrapper": "model.atst.", "bare": ""}


def _write_ckpt(path, layout, sd):
    pre = _PREFIX[layout]
    out = {}
    for k, v in sd.items():
        k2 = k
        if layout in ("lightning_teacher", "downstream_finetuned", "c2f_frame_encoder") and k.startswith("norm_frame."):
            k2 = "norm." + k[len("norm_frame."):]
        out[pre + k2] = v
    if layout == "lightning_teacher":
        out.update({"model.student.encoder." + k: v for k, v in sd.items()})
    torch.save({"state_dict": out, "hyper_parameters": {"pos_type": "cut"}}, path)


def test_loader_layouts_and_failures():
    net = rand_encoder(1); sd = net.state_dict(); x = AC.ATSTMel()(torch.randn(1, 32000) * 0.1)
    with torch.inference_mode():
        ref = net(x, torch.tensor([50]))
    with tempfile.TemporaryDirectory() as d:
        for layout in _PREFIX:
            p = os.path.join(d, layout + ".ckpt"); _write_ckpt(p, layout, sd)
            enc = AC.load_atst_encoder(p, verbose=False)
            assert enc.meta["layout"] == layout, (layout, enc.meta["layout"])
            with torch.inference_mode():
                assert torch.allclose(enc.net(x, torch.tensor([50])), ref, atol=1e-6), layout
            assert not any(q.requires_grad for q in enc.parameters())
        bad = {k: v for k, v in sd.items() if "blocks.5." not in k}
        p = os.path.join(d, "bad.ckpt"); _write_ckpt(p, "bare", bad)
        try:
            AC.load_atst_encoder(p, verbose=False); raise AssertionError("incomplete checkpoint was accepted")
        except RuntimeError:
            pass
        p = os.path.join(d, "junk.ckpt"); torch.save({"state_dict": {"foo.bar": torch.zeros(2)}}, p)
        try:
            AC.load_atst_encoder(p, verbose=False); raise AssertionError("junk checkpoint accepted")
        except RuntimeError:
            pass
        try:
            AC.load_atst_encoder(os.path.join(d, "bare.ckpt"), expected_sha256="0" * 64, verbose=False)
            raise AssertionError("sha mismatch accepted")
        except RuntimeError:
            pass
    print("ok loader: 5 layouts load strictly and reproduce outputs; incomplete/junk/sha-mismatch raise")


def test_cache_roundtrip_and_resume():
    enc = AC.ATSTEncoder(rand_encoder(), AC.ATSTMel(), {"embed_dim": 768})
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    meta = {"sha256": "x", "mel_variant": "pretrain", "n_last": 1, "chunk": 250, "hop": 125, "embed_dim": 768}
    with tempfile.TemporaryDirectory() as d:
        c = AC.ATSTCache(d, "t1", meta)
        wavs = {i: torch.randn(int((1 + i) * 1.3 * AC.SR)) * 0.05 for i in range(4)}
        s1 = AC.build_cache(list(wavs), lambda i: wavs[i], c, "nat", ex, log=lambda *_: None, max_new=2)
        s2 = AC.build_cache(list(wavs), lambda i: wavs[i], c, "nat", ex, log=lambda *_: None)
        assert s1["new"] == 2 and s2["new"] == 2 and s2["skipped"] == 2
        AC.verify_cache(c, "nat", list(wavs), lambda k: wavs[k].numel(), log=lambda *_: None, full=True)
        try:
            AC.ATSTCache(d, "other_tag", meta); raise AssertionError("mixed tags accepted")
        except RuntimeError:
            pass
        assert c.load("nat", 3).dtype == np.float16
    print("ok cache: resume, shape/finite verification, tag pinning")


# ---------------------------------------------------------------- reference (audiossl) comparisons
def _load_reference(audiossl_dir):
    sys.path.insert(0, audiossl_dir)
    if "fairseq" not in sys.modules:
        for n in ("fairseq", "fairseq.data", "fairseq.data.data_utils"):
            sys.modules[n] = types.ModuleType(n)
        sys.modules["fairseq.data.data_utils"].compute_mask_indices = lambda *a, **k: None
    from audiossl.methods.atstframe.audio_transformer import FrameAST_base
    return FrameAST_base(patch_h=64, patch_w=4, drop_path_rate=0.0).eval()


def _load_into_reference(ref, state):
    miss, unexp = ref.load_state_dict(state, strict=False)
    bad_miss = [k for k in miss if "cls_token" not in k]
    assert not bad_miss and not unexp, f"reference/state mismatch: missing {bad_miss[:5]} unexpected {list(unexp)[:5]}"
    if miss:
        print("  note: reference keys not in our state (ignored):", list(miss))


def _compare_to_reference(ref, mine, mel, tok_len, n_last, atol=1e-4):
    """audiossl takes lengths in mel frames (it floors //4 internally). Try that first, then token units, and report
    which convention reproduces our (token-unit) output. One of them must match."""
    with torch.inference_mode():
        b = mine(mel, tok_len, n_last=n_last)
        tried = {}
        for name, L in (("mel-frames", tok_len * 4), ("tokens", tok_len)):
            a = ref.get_intermediate_layers(mel, L, n=n_last, scene=False)
            ok = all(torch.allclose(a[i, :int(tok_len[i])], b[i, :int(tok_len[i])], atol=atol) for i in range(len(tok_len)))
            tried[name] = float(max((a[i, :int(tok_len[i])] - b[i, :int(tok_len[i])]).abs().max() for i in range(len(tok_len))))
            if ok:
                return name, tried[name]
    raise AssertionError(f"no length convention matches the reference: max diffs {tried}")


def test_audiossl_equivalence(audiossl_dir):
    if not audiossl_dir:
        print("skip audiossl equivalence (no clone)"); return
    ref = _load_reference(audiossl_dir)
    mine = AC.FrameASTMin().eval()
    mine.requires_grad_(False)
    ref_sd = {k: v for k, v in ref.state_dict().items() if "cls_token" not in k}
    mine.load_state_dict(ref_sd, strict=True)
    mel = AC.ATSTMel()(torch.randn(2, 16000 * 8) * 0.1)             # (2,1,64,801)
    tl = torch.tensor([200, 150])
    for n_last in (1, 4, 12):
        conv, err = _compare_to_reference(ref, mine, mel, tl, n_last)
    print(f"ok equivalence with audiossl FrameAST (n_last 1/4/12, padded item). reference length unit = {conv}; max err {err:.2e}")


def test_real_vs_reference(ckpt, audiossl_dir):
    """Same real weights in both implementations, same mel, same masks."""
    if not (ckpt and audiossl_dir):
        print("skip real-vs-reference (need ckpt and audiossl clone)"); return
    enc = AC.load_atst_encoder(ckpt, device="cpu", verbose=False)
    ref = _load_reference(audiossl_dir)
    _load_into_reference(ref, {k: v.clone() for k, v in enc.net.state_dict().items()})
    t = torch.arange(16000 * 8) / 16000.0
    w = torch.stack([0.1 * torch.sin(2 * np.pi * 440 * t) + 0.02 * torch.randn_like(t),
                     0.05 * torch.randn_like(t)])
    mel = enc.mel(w)
    conv, err = _compare_to_reference(ref, enc.net, mel, torch.tensor([200, 150]), 1)
    print(f"ok REAL checkpoint: our encoder == audiossl FrameAST on the same weights (length unit {conv}, max err {err:.2e})")


def test_real_checkpoint(ckpt):
    if not ckpt:
        print("skip real checkpoint"); return
    enc = AC.load_atst_encoder(ckpt, device="cpu", verbose=True)
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    f = ex(torch.randn(8 * AC.SR) * 0.05)
    assert f.shape == (200, 768) and np.isfinite(f).all()
    print(f"real checkpoint OK: 8 s -> {f.shape}, std {f.std():.3f}, sha256 {enc.meta['sha256']}")


ALL_SYNTHETIC = ("test_rand_encoder_nontrivial", "test_mel_per_example_floor", "test_zscore_shifts_levels",
                 "test_padding_mask_equivalence", "test_chunk_plan_coverage", "test_extractor_shapes_and_chunking",
                 "test_loader_layouts_and_failures", "test_cache_roundtrip_and_resume")

In [ ]:
# ---- CELL 3: synthetic tests (no real weights). Every line must print "ok". -----------------------------------------
import atst_common as AC; importlib.reload(AC)
import test_atst_common as T; importlib.reload(T)
for _name in T.ALL_SYNTHETIC:
    getattr(T, _name)()
print("SYNTHETIC TESTS PASSED")

In [ ]:
# ---- CELL 4: reference equivalence with the REAL weights (the check that proves the re-implementation) ---------------
_A = AUDIOSSL if os.path.isdir(AUDIOSSL) else None
if _A is None:
    print("!! no audiossl clone at", AUDIOSSL, "-> reference tests skipped. Strongly recommended: git clone https://github.com/Audio-WestlakeU/audiossl there.")
T.test_audiossl_equivalence(_A)
T.test_real_vs_reference(CKPT, _A)
T.test_real_checkpoint(CKPT)
ATST_PINNED = AC.sha256_file(CKPT)
print("checkpoint sha256 (pinned for all later stages):", ATST_PINNED)

In [ ]:
# ---- CELL 5: load the integration into THIS kernel, pin the checkpoint, run the V5-side self-tests ------------------
%run -i "$HERE/atst_v5_integration.py"
ATST_EXPECTED_SHA256 = ATST_PINNED
import vaani_track_ensemble as VE; importlib.reload(VE)
atst_selftest_v5()
VE.selftest(official_combined_score, match_events, events_to_frames, decode_v5)
print("V5-SIDE SELF-TESTS PASSED | cache dir:", ATST_CACHE_DIR, "| run tag:", ATST_RUN_TAG)

In [ ]:
# ---- CELL 6: one real clip through the real encoder, plus a chunked 23.7 s clip -------------------------------------
_enc = AC.load_atst_encoder(CKPT, device=device, mel_variant=ATST_MEL_VARIANT, expected_sha256=ATST_EXPECTED_SHA256)
_ex = AC.ATSTFeatureExtractor(_enc, device)
_clip = _load_waveform(int(VAL_GOLD[0]), full_dataset, SR, WAV_CACHE_DIRS)
_f = _ex(_clip)
print(_clip.numel() / SR, "s ->", _f.shape, "| expected tokens", AC.atst_T_expected(max(_clip.numel(), SR)), "| finite", np.isfinite(_f).all(), "| std %.3f" % _f.std())
_lf = _ex(torch.cat([_clip] * 4)[: int(23.7 * SR)])
print("23.7 s chunked ->", _lf.shape, "| finite", np.isfinite(_lf).all())
print(json.dumps(_enc.meta, indent=1)); print(AC.PROVENANCE["atst_as2M.ckpt"])
del _enc, _ex
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ---- CELL 7: storage estimate + probe cache (resumable; safe to interrupt and re-run) -------------------------------
if GO_PROBE_CACHE:
    atst_estimate_storage()
    atst_build_caches(subset="probe")
else:
    print("skipped (GO_PROBE_CACHE=False)")

In [ ]:
# ---- CELL 8: go/no-go probe. Continue to cell 10 only if "both - BEATs" has CI90 lower bound > 0 ---------------------
if GO_PROBE:
    PROBE = atst_probe(atst_open_cache(create=False), n_updates=1500, n_boot=100)
else:
    print("skipped (GO_PROBE=False)")

In [ ]:
# ---- CELL 9: Experiment A, V5-A + V5-C track ensemble (CPU only, reads real v5_work pickles) -----------------------
if GO_ENSEMBLE:
    ENS = atst_ensemble_experiment("v5a", "v5c_smel", write_zip=False, control="best", v5_dir=V5_DIR)
else:
    print("skipped (GO_ENSEMBLE=False)")

In [ ]:
# ---- CELL 10: full ATST cache (train + val + synthetic + test). Resumable. -----------------------------------------
if GO_FULL_CACHE:
    atst_estimate_storage()
    atst_build_caches(subset="all")
else:
    print("skipped (GO_FULL_CACHE=False)")

In [ ]:
# ---- CELL 11: B3 training: V5-C recipe + zero-initialised gated ATST residual. Refuses to overwrite an existing tag. --
if GO_TRAIN:
    FUSION_PATHS, FUSION_BEST = atst_stage_train_fusion("fuse", ATST_RUN_TAG)
else:
    print("skipped (GO_TRAIN=False)")

In [ ]:
# ---- CELL 12: B3 evaluation under the FIXED V5-A decoder, paired against V5-C on the same Val-N / Val-S clips --------
if GO_EVAL:
    _c = atst_open_cache(create=False)
    atst_eval_and_decoder("fuse", ATST_RUN_TAG, _c, select=False)
    _d = json.load(open(f"{V5_DIR}/v5a_decoder.json"))["decoder"]
    ENS_B3 = VE.run_ensemble_experiment(dict(globals()), "v5c_smel", ATST_RUN_TAG, ens_tag=ATST_ENS_TAG,
                                        decoders={"A": _d, "B": _d}, primary="A", control="A only", write_zip=False, log=log)
else:
    print("skipped (GO_EVAL=False)")

In [ ]:
# ---- CELL 13: B3 test tracks (float32) + zip + archive verification. Submit ONLY if cell 12 printed ACCEPT. ---------
if GO_TEST_ZIP:
    _c = atst_open_cache(create=False)
    atst_test_tracks32("fuse", ATST_RUN_TAG, _c)
    ZIP_B3 = atst_write_submission(ATST_RUN_TAG)
    atst_verify_zip(ZIP_B3, expected_ids={os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES})
    print("Candidate zip:", ZIP_B3)
else:
    print("skipped (GO_TEST_ZIP=False)")